# Realty Checker Demo

Этот notebook показывает, как пользователь может взаимодействовать с моделью.

Система получает параметры квартиры, предсказывает справедливую цену за квадратный метр и сравнивает её с фактической ценой объявления.

На выходе формируется вердикт:

- `overpriced` — квартира выглядит переоцененной
- `underpriced` — квартира выглядит недооцененной
- `fair` — цена близка к рыночной

## 1. Импорт библиотек и подключение проекта

Подключаем код проекта, чтобы использовать обученную модель и сервис `PriceChecker`.

In [3]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.services.price_checker import PriceChecker

## 2. Признаки модели

Модель использует числовые и категориальные признаки квартиры.

Часть признаков вводится пользователем, а часть создается автоматически внутри `build_features`.

In [4]:
num_features = [
    "rooms",
    "area_total",
    "area_kitchen",
    "area_living",
    "metro_time_min",
    "floor",
    "total_floors",
    "year_built",
    "completion_year",
    "photo_count",
    "is_studio",
    "has_balcony",
    "has_elevator",
    "has_parking",
    "has_renovation",
    "kitchen_ratio",
    "living_ratio",
    "floor_ratio",
    "is_first_floor",
    "is_last_floor",
    "house_age",
    "is_new_building",
]

cat_features = [
    "okrug",
    "district",
    "metro",
    "house_type",
    "seller_type",
]

## 3. Загрузка модели

Загружаем лучшую модель CatBoost, полученную после feature engineering и подбора гиперпараметров через Optuna.

In [5]:
checker = PriceChecker(
    model_path="../models/catboost_optuna.cbm",
    num_features=num_features,
    cat_features=cat_features,
    threshold_percent=10.0,
)

## 4. Пример объявления

Ниже задаются параметры квартиры.

Пользователь может менять значения в этом словаре и перезапускать следующие ячейки.

In [ ]:
listing = {
    "price": 12_000_000,  # Цена в рублях 

    "rooms": 1,  # Количество комнат
    "area_total": 40.0,  # Общая площадь квартиры в м²
    "area_kitchen": 10.0,  # Площадь кухни в м²
    "area_living": 18.0,  # Жилая площадь (комнаты) в м²
    "metro_time_min": 10,  # Время пешком до метро в минутах
    "floor": 6,  # Этаж квартиры
    "total_floors": 16,  # Всего этажей в доме

    "year_built": None,  # Год постройки дома
    "completion_year": None,  # Год сдачи дома 
    "photo_count": 0,  # Количество фотографий в объявлении

    "is_studio": 0,  # Студия 
    "has_balcony": 0,  # Есть балкон/лоджия
    "has_elevator": 0,  # Есть лифт в доме
    "has_parking": 0,  # Есть парковка
    "has_renovation": 0,  # Сделан ремонт

    "okrug": "СВАО",  # Административный округ
    "district": "Отрадное",  # Район города
    "metro": "Отрадное",  # Ближайшая станция метро
    "house_type": "unknown",  # Тип дома 
    "seller_type": "unknown",  # Тип продавца 
}

## 5. Проверка объявления

Модель предсказывает справедливую цену за квадратный метр.

Затем фактическая цена за квадратный метр сравнивается с предсказанием модели.

In [7]:
result = checker.check_price(listing)
result

{'actual_price': 12000000,
 'area_total': 40.0,
 'actual_price_per_m2': 300000.0,
 'predicted_price_per_m2': 308194.33,
 'diff_percent': -2.66,
 'verdict': 'fair',
 'comment': 'Цена выглядит близкой к рыночной.'}

## 6. Итоговый отчет

Выводим результат в понятном для пользователя формате.

In [10]:
print("REALTY CHECKER — ОТЧЕТ ПО ОБЪЯВЛЕНИЮ")

print(f"Цена квартиры: {result['actual_price']:,.0f} руб.".replace(",", " "))
print(f"Площадь: {result['area_total']} м²")

print()
print("Оценка цены:")
print(f"- Реальная цена за м²: {result['actual_price_per_m2']:,.0f} руб.".replace(",", " "))
print(f"- Оценка модели за м²: {result['predicted_price_per_m2']:,.0f} руб.".replace(",", " "))
print(f"- Отклонение: {result['diff_percent']:.2f}%")

print()
print("Вердикт:")

if result["verdict"] == "overpriced":
    print("Объявление выглядит переоцененным.")
elif result["verdict"] == "underpriced":
    print("Объявление выглядит недооцененным.")
else:
    print("Цена близка к рыночной.")

print()
print("Комментарий:")
print(result["comment"])

REALTY CHECKER — ОТЧЕТ ПО ОБЪЯВЛЕНИЮ
Цена квартиры: 12 000 000 руб.
Площадь: 40.0 м²

Оценка цены:
- Реальная цена за м²: 300 000 руб.
- Оценка модели за м²: 308 194 руб.
- Отклонение: -2.66%

Вердикт:
Цена близка к рыночной.

Комментарий:
Цена выглядит близкой к рыночной.


## 7. Готовые сценарии для проверки

Ниже приведены несколько примеров объявлений:

1. квартира с рыночной ценой
2. переоцененная квартира
3. недооцененная квартира

In [11]:
examples = {
    "fair": {
        "price": 12_000_000,
        "rooms": 1,
        "area_total": 40.0,
        "area_kitchen": 10.0,
        "area_living": 18.0,
        "metro_time_min": 10,
        "floor": 6,
        "total_floors": 16,
        "year_built": None,
        "completion_year": None,
        "photo_count": 0,
        "is_studio": 0,
        "has_balcony": 0,
        "has_elevator": 0,
        "has_parking": 0,
        "has_renovation": 0,
        "okrug": "СВАО",
        "district": "Отрадное",
        "metro": "Отрадное",
        "house_type": "unknown",
        "seller_type": "unknown",
    },

    "overpriced": {
        "price": 30_000_000,
        "rooms": 1,
        "area_total": 35.0,
        "area_kitchen": 8.0,
        "area_living": 20.0,
        "metro_time_min": 12,
        "floor": 5,
        "total_floors": 12,
        "year_built": None,
        "completion_year": None,
        "photo_count": 0,
        "is_studio": 0,
        "has_balcony": 0,
        "has_elevator": 0,
        "has_parking": 0,
        "has_renovation": 0,
        "okrug": "ЮАО",
        "district": "Орехово-Борисово",
        "metro": "Домодедовская",
        "house_type": "unknown",
        "seller_type": "unknown",
    },

    "underpriced": {
        "price": 9_000_000,
        "rooms": 2,
        "area_total": 45.0,
        "area_kitchen": 9.0,
        "area_living": 28.0,
        "metro_time_min": 8,
        "floor": 7,
        "total_floors": 14,
        "year_built": None,
        "completion_year": None,
        "photo_count": 0,
        "is_studio": 0,
        "has_balcony": 0,
        "has_elevator": 0,
        "has_parking": 0,
        "has_renovation": 0,
        "okrug": "ЗАО",
        "district": "Кунцево",
        "metro": "Молодежная",
        "house_type": "unknown",
        "seller_type": "unknown",
    },
}

## 8. Проверка нескольких сценариев

Запускаем модель на нескольких заранее подготовленных примерах.

In [13]:
for name, example in examples.items():
    result = checker.check_price(example)

    print()
    print(f"Сценарий: {name}")
    print()

    print(f"Цена квартиры: {result['actual_price']:,.0f} руб.".replace(",", " "))
    print(f"Площадь: {result['area_total']} м²")
    print(f"Реальная цена за м²: {result['actual_price_per_m2']:,.0f} руб.".replace(",", " "))
    print(f"Оценка модели за м²: {result['predicted_price_per_m2']:,.0f} руб.".replace(",", " "))
    print(f"Отклонение: {result['diff_percent']:.2f}%")
    print(f"Вердикт: {result['verdict']}")
    print(f"Комментарий: {result['comment']}")
    print()


Сценарий: fair

Цена квартиры: 12 000 000 руб.
Площадь: 40.0 м²
Реальная цена за м²: 300 000 руб.
Оценка модели за м²: 308 194 руб.
Отклонение: -2.66%
Вердикт: fair
Комментарий: Цена выглядит близкой к рыночной.


Сценарий: overpriced

Цена квартиры: 30 000 000 руб.
Площадь: 35.0 м²
Реальная цена за м²: 857 143 руб.
Оценка модели за м²: 246 363 руб.
Отклонение: 247.92%
Вердикт: overpriced
Комментарий: Цена выглядит завышенной относительно модели.


Сценарий: underpriced

Цена квартиры: 9 000 000 руб.
Площадь: 45.0 м²
Реальная цена за м²: 200 000 руб.
Оценка модели за м²: 422 635 руб.
Отклонение: -52.68%
Вердикт: underpriced
Комментарий: Цена выглядит заниженной относительно модели.

